# ROGUE (1980) — 코랩판

세 파일로 나뉘어 있습니다.

| 파일 | 역할 |
|---|---|
| `rogue_core.py` | 상수 · 클래스 · 랜덤 던전 생성 |
| `rogue_game.py` | 게임 규칙 (이동 · 전투 · 턴 · 층) |
| 이 노트북 | 화면 그리기 · 키보드 · 실행 |

**플레이 방법**: 아래 세 셀을 위에서부터 차례로 실행 → 게임 화면을 한 번 클릭 → 키보드로 조작.

```
@ 플레이어    r g o T 몬스터    ! 물약    $ 금화
> 계단        ^ 터진 함정       * 최종 보물
```

`W A S D` 이동/공격 · `G` 줍기 · `U` 물약 · `I` 가방 · `>` 계단 · `.` 대기 · `Q` 종료 · `R` 재시작

5층에서 `*` 위에 올라가 `G` 를 누르면 클리어.

## 1. 모듈 파일 준비

In [ ]:
# 1) rogue_core.py, rogue_game.py 를 /content 에 올린다
#    - 왼쪽 파일 탭에 드래그해도 되고, 아래 업로드 창을 써도 된다
import os

need = [f for f in ("rogue_core.py", "rogue_game.py") if not os.path.exists(f)]

if need:
    print("다음 파일이 없습니다:", need)
    from google.colab import files
    files.upload()
else:
    print("모듈 파일 확인 완료:", os.listdir("."))

## 2. 화면 렌더러

In [ ]:
# 2) 모듈 불러오기 + 화면 그리기
import importlib
import html

import rogue_core
import rogue_game

# 파일을 수정한 뒤 이 셀만 다시 실행하면 바로 반영된다
importlib.reload(rogue_core)
importlib.reload(rogue_game)

from rogue_core import MAP_WIDTH, MAP_HEIGHT, MAX_FLOOR
from rogue_game import Game, MAX_MESSAGES

SCREEN_WIDTH = MAP_WIDTH


def render(game):
    """게임 상태를 하나의 문자열 화면으로 만든다."""
    p = game.player
    lines = []

    lines.append("=" * SCREEN_WIDTH)
    lines.append("ASCII ROGUE".center(SCREEN_WIDTH))
    lines.append(f"FLOOR {game.floor} / {MAX_FLOOR}".center(SCREEN_WIDTH))
    lines.append("=" * SCREEN_WIDTH)

    # ----- 던전 -----
    for y in range(MAP_HEIGHT):
        row = []

        for x in range(MAP_WIDTH):

            if not game.is_explored(x, y):
                row.append(" ")
                continue

            if x == p.x and y == p.y:
                row.append("@")
                continue

            monster = game.monster_at(x, y)
            if monster:
                row.append(monster.char)
                continue

            item = game.item_at(x, y)
            if item:
                row.append(item.symbol)
                continue

            trap = game.trap_at(x, y)
            if trap and trap.visible:
                row.append(trap.symbol)
                continue

            if game.stairs and x == game.stairs.x and y == game.stairs.y:
                row.append(game.stairs.symbol)
                continue

            if game.treasure and x == game.treasure.x and y == game.treasure.y:
                row.append(game.treasure.symbol)
                continue

            row.append(game.dungeon[y][x])

        lines.append("".join(row))

    # ----- 상태 -----
    lines.append("-" * SCREEN_WIDTH)

    bar_len = 20
    filled = int(bar_len * p.hp / p.max_hp) if p.max_hp else 0
    bar = "#" * filled + "-" * (bar_len - filled)

    lines.append(f"HP [{bar}] {p.hp}/{p.max_hp}")
    lines.append(
        f"LV {p.level}  XP {p.xp}/{p.next_xp}  "
        f"ATK {p.attack}  DEF {p.defense}  "
        f"GOLD {p.gold}  TURN {game.turn}"
    )

    flags = []
    if p.poisoned:
        flags.append(f"POISON({p.poison_turns})")
    if game.is_on_stairs():
        flags.append("계단 위 [>]")
    if game.is_on_treasure():
        flags.append("보물 위 [G]")
    if flags:
        lines.append("상태: " + "   ".join(flags))

    # ----- 메시지 로그 -----
    lines.append("-" * SCREEN_WIDTH)

    messages = game.messages[-MAX_MESSAGES:]
    for message in messages:
        lines.append("> " + message)
    for _ in range(MAX_MESSAGES - len(messages)):
        lines.append("")

    # ----- 인벤토리 -----
    if game.show_inventory:
        lines.append("-" * SCREEN_WIDTH)
        lines.append("[ INVENTORY ]   (I 로 닫기)")

        if not p.inventory:
            lines.append("  비어 있음")
        else:
            counts = {}
            for item in p.inventory:
                counts[item.name] = counts.get(item.name, 0) + 1
            for name, n in counts.items():
                lines.append(f"  {name} x{n}")

    # ----- 종료 화면 -----
    if not game.running:
        lines.append("=" * SCREEN_WIDTH)
        if game.won:
            lines.append("GAME CLEAR!".center(SCREEN_WIDTH))
        elif not p.alive:
            lines.append("YOU DIED".center(SCREEN_WIDTH))
        else:
            lines.append("GAME QUIT".center(SCREEN_WIDTH))
        lines.append("R : 다시 시작".center(SCREEN_WIDTH))
        lines.append("=" * SCREEN_WIDTH)

    return "\n".join(lines)


def screen_html(game, error=None):
    """검은 배경의 터미널 화면 HTML."""
    if error is None:
        body = html.escape(render(game))
        color = "#d0ffd0"
    else:
        body = html.escape(error)
        color = "#ff6b6b"

    return f"""
    <pre style="
        margin: 0;
        padding: 16px;
        background: #000000;
        color: {color};
        border: 2px solid #444;
        border-radius: 8px;
        font-family: 'DejaVu Sans Mono', Menlo, Consolas, monospace;
        font-size: 14px;
        line-height: 1.15;
        white-space: pre;
        overflow-x: auto;
    ">{body}</pre>
    """


print("화면 렌더러 준비 완료")

## 3. 게임 실행

In [ ]:
# 3) 게임 실행 (이 셀을 다시 실행하면 새 게임)
import traceback

import ipywidgets as widgets
from IPython.display import display, Javascript
from google.colab import output

game = Game()

screen = widgets.HTML(
    value=screen_html(game),
    layout=widgets.Layout(width="720px"),
)


def do_key(key):
    """키 하나를 처리하고 화면을 다시 그린다."""
    try:
        game.handle_key(key)
        screen.value = screen_html(game)
    except Exception:
        # 콜백 안에서 난 예외는 코랩이 조용히 삼킨다.
        # 그래서 화면에 직접 찍어준다. (원래 화면이 멈추던 이유)
        screen.value = screen_html(game, error=traceback.format_exc())

    return "OK"


# 자바스크립트에서 부를 수 있게 등록
output.register_callback("rogue.key", do_key)


def key_button(label, key, width="54px"):
    button = widgets.Button(
        description=label,
        layout=widgets.Layout(width=width, height="34px"),
    )
    button.on_click(lambda _b, k=key: do_key(k))
    return button


pad = widgets.VBox([
    widgets.HBox([
        widgets.Label(layout=widgets.Layout(width="54px")),
        key_button("W", "w"),
    ]),
    widgets.HBox([
        key_button("A", "a"),
        key_button("S", "s"),
        key_button("D", "d"),
    ]),
])

actions = widgets.VBox([
    widgets.HBox([
        key_button("G 줍기", "g", "90px"),
        key_button("U 물약", "u", "90px"),
        key_button("I 가방", "i", "90px"),
    ]),
    widgets.HBox([
        key_button("> 계단", ">", "90px"),
        key_button(". 대기", ".", "90px"),
        key_button("R 재시작", "r", "90px"),
    ]),
])

display(widgets.VBox([
    screen,
    widgets.HBox([pad, widgets.Label(" "), actions]),
]))


# 키보드 연결 (게임 화면을 한 번 클릭한 뒤 키를 누르세요)
display(Javascript("""
(() => {
    if (window.rogueKeyHandler) {
        document.removeEventListener("keydown", window.rogueKeyHandler, true);
    }

    const validKeys = new Set(
        ["w", "a", "s", "d", "g", "u", "i", "q", "r", ">", ".", " "]
    );

    window.rogueKeyHandler = function (event) {
        let key = event.key.toLowerCase();

        if (key === "arrowup")    { key = "w"; }
        if (key === "arrowdown")  { key = "s"; }
        if (key === "arrowleft")  { key = "a"; }
        if (key === "arrowright") { key = "d"; }

        if (!validKeys.has(key)) { return; }

        event.preventDefault();
        event.stopPropagation();

        google.colab.kernel.invokeFunction("rogue.key", [key], {});
    };

    document.addEventListener("keydown", window.rogueKeyHandler, true);

    console.log("ROGUE KEYBOARD READY");
})();
"""))

### 잘 안 될 때

- **키가 안 먹힌다** → 게임 화면을 한 번 클릭해서 출력 영역에 포커스를 준다. 그래도 안 되면 아래 버튼으로 플레이하면 된다 (버튼은 항상 동작).
- **화면이 안 바뀐다** → 이제는 멈추는 대신 빨간 글씨로 오류 내용이 그대로 화면에 찍힌다. 그 내용을 보고 고치면 된다.
- **모듈을 고쳤는데 반영이 안 된다** → 2번 셀부터 다시 실행 (`importlib.reload` 가 들어 있다).